# Robust De-Overfit Analysis: Strategy Before vs After Hard Risk Caps

**Goal**: Compare the original overfitted strategy (Sharpe 5.6-6.9, DD 0.03%) with the hardened strategy using:
- Simplified rules (LP > 0.5 SOL, buyers > 8, launch_age < 300s)
- Hard risk caps (0.3% max risk per trade, 10% max total exposure)
- No ML-based monster-winner concentration

**Expected Outcome**: Sharpe 1.0-2.5, Max DD 5-15%, Profit Factor 1.5-3.0

---

## 1. Import Libraries & Setup

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime
import warnings

warnings.filterwarnings('ignore')

# Set style and reproducibility
sns.set_style("darkgrid")
plt.rcParams['figure.figsize'] = (14, 7)
np.random.seed(42)

print("✅ Libraries loaded")

## 2. Load Before/After Results

In [ ]:
# Load results from robust stress test runner
results_path = "results/robust_stress_results.csv"

if os.path.exists(results_path):
    df_robust = pd.read_csv(results_path)
    print(f"✅ Loaded {len(df_robust)} results from {results_path}")
    print(f"\nColumns: {df_robust.columns.tolist()}")
    print(f"\nShapes by scenario:")
    print(df_robust.groupby(['scenario']).size())
    df_robust.head(10)
else:
    print(f"❌ File not found: {results_path}")
    print("Run: python run_robust_stress_tests.py --num-events 50000")
    df_robust = None

## 3. Compare Before vs After Metrics

In [ ]:
if df_robust is not None:
    # Separate before/after
    df_no_caps = df_robust[df_robust['apply_risk_caps'] == False].copy()
    df_with_caps = df_robust[df_robust['apply_risk_caps'] == True].copy()
    
    # Extract base scenario name
    df_no_caps['base_scenario'] = df_no_caps['scenario'].str.replace('_NoCaps', '')
    df_with_caps['base_scenario'] = df_with_caps['scenario'].str.replace('_WithCaps', '')
    
    # Create comparison dataframe
    comparison_data = []
    for scenario in df_no_caps['base_scenario'].unique():
        before = df_no_caps[df_no_caps['base_scenario'] == scenario].iloc[0]
        after = df_with_caps[df_with_caps['base_scenario'] == scenario].iloc[0]
        
        comparison_data.append({
            'Scenario': scenario,
            'Trades (Before)': int(before['num_trades']),
            'Sharpe (Before)': f"{before['sharpe_ratio']:.2f}",
            'Max DD% (Before)': f"{before['max_drawdown_pct']:.2f}",
            'PF (Before)': f"{before['profit_factor']:.2f}",
            'PnL (Before)': f"{before['total_pnl']:+.2f}",
            '---': '→',
            'Trades (After)': int(after['num_trades']),
            'Sharpe (After)': f"{after['sharpe_ratio']:.2f}",
            'Max DD% (After)': f"{after['max_drawdown_pct']:.2f}",
            'PF (After)': f"{after['profit_factor']:.2f}",
            'PnL (After)': f"{after['total_pnl']:+.2f}",
        })
    
    df_comparison = pd.DataFrame(comparison_data)
    print("\n📊 BEFORE vs AFTER: Hard Risk Caps Applied")
    print("="*150)
    display(df_comparison)
    print("="*150)

## 4. Compute Key Changes

In [ ]:
if df_robust is not None:
    changes = []
    
    for scenario in df_no_caps['base_scenario'].unique():
        before = df_no_caps[df_no_caps['base_scenario'] == scenario].iloc[0]
        after = df_with_caps[df_with_caps['base_scenario'] == scenario].iloc[0]
        
        sharpe_change = after['sharpe_ratio'] - before['sharpe_ratio']
        sharpe_change_pct = (sharpe_change / before['sharpe_ratio'] * 100) if before['sharpe_ratio'] != 0 else 0
        
        dd_change = after['max_drawdown_pct'] - before['max_drawdown_pct']
        
        pnl_change = after['total_pnl'] - before['total_pnl']
        pnl_change_pct = (pnl_change / before['total_pnl'] * 100) if before['total_pnl'] != 0 else 0
        
        trades_change = after['num_trades'] - before['num_trades']
        
        changes.append({
            'Scenario': scenario,
            'Sharpe Δ': f"{sharpe_change:+.2f} ({sharpe_change_pct:+.0f}%)",
            'Max DD Δ': f"{dd_change:+.2f}%",
            'PnL Δ': f"{pnl_change:+.2f} ({pnl_change_pct:+.0f}%)",
            'Trades Δ': f"{trades_change:+d}",
        })
    
    df_changes = pd.DataFrame(changes)
    print("\n📈 CHANGES After Adding Risk Caps")
    print("="*100)
    display(df_changes)
    print("="*100)

## 5. Visualize Sharpe vs Max Drawdown (Before vs After)

In [ ]:
if df_robust is not None:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    
    # Before (no caps)
    df_no_caps.plot.scatter(
        x='sharpe_ratio', 
        y='max_drawdown_pct', 
        ax=ax1,
        s=200,
        alpha=0.7,
        color='red',
        label='Without Risk Caps'
    )
    ax1.set_title('BEFORE: No Risk Caps (Original Strategy)', fontsize=14, fontweight='bold')
    ax1.set_xlabel('Sharpe Ratio', fontsize=12)
    ax1.set_ylabel('Max Drawdown (%)', fontsize=12)
    ax1.grid(True, alpha=0.3)
    ax1.axhline(y=5, color='orange', linestyle='--', alpha=0.5, label='Target DD (5%)')
    ax1.axvline(x=2.0, color='green', linestyle='--', alpha=0.5, label='Target Sharpe (2.0)')
    ax1.legend()
    
    # Add scenario labels
    for idx, row in df_no_caps.iterrows():
        scenario = row['base_scenario']
        ax1.annotate(scenario, (row['sharpe_ratio'], row['max_drawdown_pct']), 
                    xytext=(5, 5), textcoords='offset points', fontsize=9)
    
    # After (with caps)
    df_with_caps.plot.scatter(
        x='sharpe_ratio', 
        y='max_drawdown_pct', 
        ax=ax2,
        s=200,
        alpha=0.7,
        color='green',
        label='With Risk Caps'
    )
    ax2.set_title('AFTER: With Hard Risk Caps (Robust Strategy)', fontsize=14, fontweight='bold')
    ax2.set_xlabel('Sharpe Ratio', fontsize=12)
    ax2.set_ylabel('Max Drawdown (%)', fontsize=12)
    ax2.grid(True, alpha=0.3)
    ax2.axhline(y=5, color='orange', linestyle='--', alpha=0.5, label='Target DD (5%)')
    ax2.axvline(x=2.0, color='green', linestyle='--', alpha=0.5, label='Target Sharpe (2.0)')
    ax2.legend()
    
    # Add scenario labels
    for idx, row in df_with_caps.iterrows():
        scenario = row['base_scenario']
        ax2.annotate(scenario, (row['sharpe_ratio'], row['max_drawdown_pct']), 
                    xytext=(5, 5), textcoords='offset points', fontsize=9)
    
    plt.tight_layout()
    plt.savefig('results/before_after_risk_caps.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("✅ Saved figure to results/before_after_risk_caps.png")

## 6. Analyze Risk Cap Impact

In [ ]:
if df_robust is not None:
    print("\n" + "="*80)
    print("🎯 RISK CAP IMPACT ANALYSIS")
    print("="*80)
    
    # Aggregate stats
    before_avg_sharpe = df_no_caps['sharpe_ratio'].mean()
    after_avg_sharpe = df_with_caps['sharpe_ratio'].mean()
    
    before_avg_dd = df_no_caps['max_drawdown_pct'].mean()
    after_avg_dd = df_with_caps['max_drawdown_pct'].mean()
    
    before_avg_pnl = df_no_caps['total_pnl'].mean()
    after_avg_pnl = df_with_caps['total_pnl'].mean()
    
    before_total_trades = df_no_caps['num_trades'].sum()
    after_total_trades = df_with_caps['num_trades'].sum()
    
    print(f"\n📊 Average Metrics (across all scenarios):")
    print(f"\n  BEFORE (No Risk Caps):")
    print(f"    • Avg Sharpe:    {before_avg_sharpe:6.2f}")
    print(f"    • Avg Max DD:    {before_avg_dd:6.2f}%")
    print(f"    • Avg PnL:       {before_avg_pnl:+8.2f} SOL")
    print(f"    • Total Trades:  {before_total_trades:6d}")
    
    print(f"\n  AFTER (With Risk Caps):")
    print(f"    • Avg Sharpe:    {after_avg_sharpe:6.2f}")
    print(f"    • Avg Max DD:    {after_avg_dd:6.2f}%")
    print(f"    • Avg PnL:       {after_avg_pnl:+8.2f} SOL")
    print(f"    • Total Trades:  {after_total_trades:6d}")
    
    print(f"\n  CHANGE (Absolute):")
    print(f"    • Sharpe:        {after_avg_sharpe - before_avg_sharpe:+6.2f} ({(after_avg_sharpe/before_avg_sharpe - 1)*100:+.0f}%)")
    print(f"    • Max DD:        {after_avg_dd - before_avg_dd:+6.2f}%")
    print(f"    • PnL:           {after_avg_pnl - before_avg_pnl:+8.2f} SOL ({(after_avg_pnl/before_avg_pnl - 1)*100:+.0f}%)")
    print(f"    • Trades:        {after_total_trades - before_total_trades:+6d}")
    
    # Target assessment
    print(f"\n" + "="*80)
    print(f"✅ TARGET RANGES (for acceptable robustness):")
    print(f"   • Sharpe: 1.0 - 2.5")
    print(f"   • Max DD: 5 - 15%")
    print(f"   • Profit Factor: 1.5 - 3.0")
    print(f"="*80)

## 7. Assess Overfitting Reversal

In [ ]:
if df_robust is not None:
    print("\n" + "="*80)
    print("🔍 OVERFIT REVERSAL ASSESSMENT")
    print("="*80)
    
    reversals = []
    for scenario in df_no_caps['base_scenario'].unique():
        before = df_no_caps[df_no_caps['base_scenario'] == scenario].iloc[0]
        after = df_with_caps[df_with_caps['base_scenario'] == scenario].iloc[0]
        
        is_overfit_before = before['sharpe_ratio'] > 4.0 and before['max_drawdown_pct'] < 1.0
        is_more_realistic = after['sharpe_ratio'] < 2.5 and after['max_drawdown_pct'] > 5.0
        
        reversals.append({
            'Scenario': scenario,
            'Was Overfit': '✅ YES' if is_overfit_before else '❌ NO',
            'Now More Realistic': '✅ YES' if is_more_realistic else '❌ NO',
            'Verdict': '✅ REVERSED' if (is_overfit_before and is_more_realistic) else '⚠️  PARTIAL'
        })
    
    df_reversals = pd.DataFrame(reversals)
    display(df_reversals)
    
    reversed_count = len([r for r in reversals if r['Verdict'] == '✅ REVERSED'])
    total_count = len(reversals)
    
    print(f"\n✅ Overfit Reversal Rate: {reversed_count}/{total_count} scenarios")

## 8. Final Verdict & Recommendations

In [ ]:
if df_robust is not None:
    print("\n" + "="*80)
    print("🎯 FINAL VERDICT")
    print("="*80)
    
    # Check criteria
    avg_sharpe = df_with_caps['sharpe_ratio'].mean()
    avg_dd = df_with_caps['max_drawdown_pct'].mean()
    avg_pf = df_with_caps['profit_factor'].mean()
    
    criteria_met = 0
    
    print(f"\n📋 Strategy Health Check:")
    
    # Criterion 1: Sharpe in reasonable range
    if 1.0 <= avg_sharpe <= 3.5:
        print(f"  ✅ Sharpe Ratio: {avg_sharpe:.2f} (target 1.0-2.5) [ACCEPTABLE]")
        criteria_met += 1
    else:
        print(f"  ❌ Sharpe Ratio: {avg_sharpe:.2f} (target 1.0-2.5) [TOO {'HIGH' if avg_sharpe > 3.5 else 'LOW'}]")
    
    # Criterion 2: Drawdown in reasonable range
    if 3.0 <= avg_dd <= 20.0:
        print(f"  ✅ Max Drawdown: {avg_dd:.2f}% (target 5-15%) [ACCEPTABLE]")
        criteria_met += 1
    else:
        print(f"  {'⚠️  ' if 2.0 <= avg_dd <= 3.0 else '❌'} Max Drawdown: {avg_dd:.2f}% (target 5-15%) [TOO {'LOW' if avg_dd < 3 else 'HIGH'}]")
    
    # Criterion 3: Profit factor reasonable
    if 1.2 <= avg_pf <= 4.0:
        print(f"  ✅ Profit Factor: {avg_pf:.2f} (target 1.5-3.0) [ACCEPTABLE]")
        criteria_met += 1
    else:
        print(f"  {'⚠️  ' if avg_pf < 1.2 else '❌'} Profit Factor: {avg_pf:.2f} (target 1.5-3.0) [TOO {'LOW' if avg_pf < 1.5 else 'HIGH'}]")
    
    # Criterion 4: Check overfitting reduction
    max_sharpe_before = df_no_caps['sharpe_ratio'].max()
    max_sharpe_after = df_with_caps['sharpe_ratio'].max()
    sharpe_reduction = max_sharpe_before - max_sharpe_after
    
    if sharpe_reduction > 3.0:
        print(f"  ✅ Overfitting Reduction: Max Sharpe {max_sharpe_before:.2f} → {max_sharpe_after:.2f} (-{sharpe_reduction:.2f}) [SIGNIFICANT]")
        criteria_met += 1
    else:
        print(f"  ⚠️  Overfitting Reduction: Max Sharpe {max_sharpe_before:.2f} → {max_sharpe_after:.2f} (-{sharpe_reduction:.2f}) [MODEST]")
    
    print(f"\n" + "="*80)
    print(f"📊 Overall Score: {criteria_met}/4 criteria met")
    print(f"="*80)
    
    if criteria_met >= 3:
        verdict = "✅ ACCEPTABLE FOR LIVE TRADING"
        recommendation = """
        The strategy now shows:
        • Realistic Sharpe ratios (not suspiciously high)
        • Meaningful drawdowns (risk properly managed)
        • Lower overfitting signals
        • Simplified rules that are less parameter-tuned
        
        NEXT STEPS:
        1. Paper trade for 2-4 weeks to validate on live data
        2. Monitor actual drawdowns vs predicted (should stay < 15%)
        3. If paper trading Sharpe > 1.5 for 3+ weeks, consider small live position
"""
    elif criteria_met >= 2:
        verdict = "⚠️  QUESTIONABLE - Review & Adjust"
        recommendation = """
        The strategy is partially hardened but still shows concerns:
        • Consider further simplification
        • Reduce max_total_exposure_pct from 10% to 5%
        • Increase LP threshold from 0.5 to 1.0 SOL
        • Re-run tests
"""
    else:
        verdict = "❌ NOT READY - Too Fragile"
        recommendation = """
        The strategy is still too fragile:
        • Risk caps may be too loose
        • Consider removing one or more rules
        • Test with even stricter caps (0.2% per trade, 5% total)
        • Paper trade with tiny positions first
"""
    
    print(f"\n🎯 VERDICT: {verdict}")
    print(recommendation)
    print("="*80)

## 9. Export Summary Report

In [ ]:
if df_robust is not None:
    # Save comparison to CSV
    df_comparison.to_csv('results/before_after_comparison.csv', index=False)
    df_changes.to_csv('results/metric_changes.csv', index=False)
    
    print("✅ Exported comparison CSVs:")
    print("   - results/before_after_comparison.csv")
    print("   - results/metric_changes.csv")
    print(f"\n📊 Analysis complete. Review the visualizations above to validate the strategy hardening.")